# Aula 15 - Notebook: Simulação de Injeção de Falhas e Desvio Automático de Fluxo

Neste notebook simulamos uma contingência operacional crítica em malha fechada na **Planta Piloto de Produção de Biodiesel por Transesterificação em Batelada (Grupo 5)**.

### Objetivos Práticos:
1. Modelar a rede hidráulica de transferência de reagentes e produtos como um grafo dirigido dinâmico $G_t = (V, E_t, W_t)$;
2. Simular a ocorrência de vazamento súbito na tubulação direta do reator (`R-200 -> TK-300_Decant`, válvula ISA `XV-203`);
3. Executar o operador de punição métrica topológica ($W \leftarrow \infty$) sem desalocar estruturas da memória do CLP;
4. Recalcular em frações de milissegundo a nova rota de contingência via tanque pulmão/arrefecimento (`TK-201_Pulmao`) utilizando o Algoritmo de Dijkstra com Min-Heap;
5. Mensurar o tempo de decisão algorítmica (`Tempo_Decisão_ms`) frente aos requisitos da norma IEC 61511 (SIL 3).


## 1. Funções de Formatação ASCII e Classe Base do Grafo

Implementação das funções auxiliares para exibição de relatórios em console industrial e da classe orientada a objetos `GrafoTubulacao`.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura padrao SCADA."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D numerica em tabela ASCII com alinhamento a direita."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoTubulacao:
    """Modelagem matematica de malha de tubulacoes industriais como digrafo ponderado."""
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): 
            self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_tubulacao(self, origem, destino, comprimento_m, tag_valvula, diametro_pol=4.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem, 
            "Destino": destino,
            "Comprimento (m)": comprimento_m, 
            "Válvula ISA": tag_valvula, 
            "Diâmetro (pol)": diametro_pol
        })


## 2. Construção da Topologia da Planta de Biodiesel (Grupo 5)

Instanciação da rede com os 8 equipamentos do núcleo de reação e decantação e os 9 trechos de duto instrumentados segundo a norma ISA-5.1.


In [2]:
def criar_rede_biodiesel() -> GrafoTubulacao:
    """Instancia a malha operacional da Planta de Biodiesel (Grupo 5)."""
    nos = ["TK-101_Oleo", "TK-103_Mix", "MAN-101", "P-101", "P-102", "R-200", "TK-201_Pulmao", "TK-300_Decant"]
    g = GrafoTubulacao(nos)
    # Segmentos de carga de reagentes
    g.adicionar_tubulacao("TK-101_Oleo", "MAN-101", 15.0, "XV-101", 4.0)
    g.adicionar_tubulacao("TK-103_Mix", "MAN-101", 12.0, "XV-103", 3.0)
    # Segmentos de succao de bombas redundantes
    g.adicionar_tubulacao("MAN-101", "P-101", 8.0, "XV-101A", 4.0)
    g.adicionar_tubulacao("MAN-101", "P-102", 10.0, "XV-101B", 4.0)
    # Segmentos de recalque para o reator CSTR encamisado
    g.adicionar_tubulacao("P-101", "R-200", 25.0, "XV-201A", 4.0)
    g.adicionar_tubulacao("P-102", "R-200", 22.0, "XV-201B", 4.0)
    # Segmentos de descarga pos-reacao e arrefecimento
    g.adicionar_tubulacao("R-200", "TK-300_Decant", 30.0, "XV-203", 6.0)
    g.adicionar_tubulacao("R-200", "TK-201_Pulmao", 18.0, "XV-202B", 6.0)
    g.adicionar_tubulacao("TK-201_Pulmao", "TK-300_Decant", 20.0, "XV-204", 6.0)
    return g

import time
import heapq
from typing import Dict, Any, Tuple, Optional, Set, List

rede = criar_rede_biodiesel()


## 3. Roteador de Menor Caminho com Fila de Prioridades Min-Heap

Implementação do algoritmo guloso de Dijkstra com suporte a bloqueio dinâmico de equipamentos e reconstrução linear de rotas.


In [3]:
class RoteadorDijkstra:
    """Motor de calculo de caminho minimo (SSSP) com Min-Heap binario (heapq)."""
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def calcular_menor_caminho(self, origem: str, destino: str, 
                               bloqueios: Optional[Set[str]] = None) -> Tuple[float, List[str]]:
        """Calcula a rota otima entre origem e destino desconsiderando nos bloqueados."""
        if bloqueios is None: 
            bloqueios = set()
        if origem in bloqueios or destino in bloqueios: 
            return float('inf'), []
        
        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        
        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]: 
                continue
            if u == destino: 
                break
            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                v = self.g.idx_to_v[v_idx]
                peso = self.g.adj_pesos[u_idx][v_idx]
                if peso < float('inf') and v not in bloqueios:
                    nova_d = d_u + peso
                    if nova_d < dist[v]:
                        dist[v] = nova_d
                        pred[v] = u
                        heapq.heappush(heap, (nova_d, v))
                        
        # Reconstrucao do caminho por retropropagacao de predecessores
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        caminho.reverse()
        
        if caminho and caminho[0] == origem:
            return dist[destino], caminho
        return float('inf'), []


## 4. Sistema Supervisório de Desvio Automático em Tempo Real

A classe `SistemaDesvioAutomatico` integra o isolamento por punição métrica com a medição de alta precisão do tempo de decisão computacional.


In [4]:
class SistemaDesvioAutomatico:
    """Modulo SCADA responsavel por isolamento de vazamentos e comutacao automatica."""
    def __init__(self, grafo: GrafoTubulacao):
        self.grafo = grafo
        self.roteador = RoteadorDijkstra(grafo)
        
    def tratar_evento_vazamento(self, origem_vaz: str, destino_vaz: str,
                                origem_fluxo: str, destino_fluxo: str) -> Dict[str, Any]:
        """Isola o segmento de duto avariado e recalcula a rota otima remanescente."""
        t0 = time.perf_counter()
        u = self.grafo.v_to_idx[origem_vaz]
        v = self.grafo.v_to_idx[destino_vaz]
        
        # Aplicacao do operador de punicao topologica de seguranca
        self.grafo.adj_pesos[u][v] = float('inf')
        self.grafo.adj_binaria[u][v] = 0
        
        # Recalculo instantaneo via Dijkstra
        novo_custo, nova_rota = self.roteador.calcular_menor_caminho(origem_fluxo, destino_fluxo)
        t_ms = (time.perf_counter() - t0) * 1000.0
        
        rota_str = " -> ".join(nova_rota) if nova_rota else "SEM ROTA VIAVEL (ESD DISPARADO)"
        custo_str = f"{novo_custo:.1f}" if novo_custo < float('inf') else "INDEFINIDO"
        
        return {
            "Trecho_Isolado": f"{origem_vaz} -> {destino_vaz}",
            "Nova_Rota_Ativa": rota_str,
            "Comprimento_Total_m": custo_str,
            "Tempo_Decisão_ms": f"{t_ms:.4f}"
        }


## 5. Simulação de Testes Operacionais e Validação de Falhas

Execução da simulação de contingência com validação de desvios e testes de estresse na malha da Planta de Biodiesel.


In [5]:
desviador = SistemaDesvioAutomatico(rede)

# 1. Validacao da rota nominal antes da falha
custo_nom, rota_nom = desviador.roteador.calcular_menor_caminho("TK-101_Oleo", "TK-300_Decant")
print("1. ROTA NOMINAL PRE-FALHA (Oleo Vegetal -> Decantador):")
print(f"Rota: {' -> '.join(rota_nom)} | Custo: {custo_nom:.1f} m\n")

# 2. Injeção de vazamento no trecho direto do reator (XV-203)
res_vaz = desviador.tratar_evento_vazamento("R-200", "TK-300_Decant", "TK-101_Oleo", "TK-300_Decant")
print("=== RELATÓRIO DE DESVIO AUTOMÁTICO DE FLUIDOS (CONTINGÊNCIA R-200 -> TK-300) ===")
print(formatar_tabela([res_vaz]))
print()

# 3. Verificacao de contingencia com alimentacao de Metoxido de Sodio (TK-103_Mix)
res_mix = desviador.tratar_evento_vazamento("R-200", "TK-300_Decant", "TK-103_Mix", "TK-300_Decant")
print("=== RELATÓRIO DE CONTINGÊNCIA PARA CARGA DE METÓXIDO (TK-103_Mix) ===")
print(formatar_tabela([res_mix]))
print()

# 4. Simulacao de Falha Dupla Catastrofica (Ruptura adicional no Pulmao XV-204)
res_esd = desviador.tratar_evento_vazamento("TK-201_Pulmao", "TK-300_Decant", "TK-101_Oleo", "TK-300_Decant")
print("=== RELATÓRIO DE FALHA DUPLA SIMULTÂNEA (ESD / PARADA DE EMERGÊNCIA) ===")
print(formatar_tabela([res_esd]))
print()

# Assercoes formais de engenharia
assert custo_nom == 77.0
assert rota_nom == ["TK-101_Oleo", "MAN-101", "P-102", "R-200", "TK-300_Decant"]
assert "TK-201_Pulmao" in res_vaz["Nova_Rota_Ativa"]
assert res_vaz["Comprimento_Total_m"] == "85.0"
assert res_vaz["Nova_Rota_Ativa"] == "TK-101_Oleo -> MAN-101 -> P-102 -> R-200 -> TK-201_Pulmao -> TK-300_Decant"
assert res_mix["Comprimento_Total_m"] == "82.0"
assert res_esd["Comprimento_Total_m"] == "INDEFINIDO"
print("TODAS AS ASSERÇÕES DE ENGENHARIA FORAM VALIDADAS COM SUCESSO!")


1. ROTA NOMINAL PRE-FALHA (Oleo Vegetal -> Decantador):
Rota: TK-101_Oleo -> MAN-101 -> P-102 -> R-200 -> TK-300_Decant | Custo: 77.0 m

=== RELATÓRIO DE DESVIO AUTOMÁTICO DE FLUIDOS (CONTINGÊNCIA R-200 -> TK-300) ===
Trecho_Isolado         | Nova_Rota_Ativa                                                              | Comprimento_Total_m | Tempo_Decisão_ms
-----------------------+------------------------------------------------------------------------------+---------------------+-----------------
R-200 -> TK-300_Decant | TK-101_Oleo -> MAN-101 -> P-102 -> R-200 -> TK-201_Pulmao -> TK-300_Decant | 85.0                | 0.0480          

=== RELATÓRIO DE CONTINGÊNCIA PARA CARGA DE METÓXIDO (TK-103_Mix) ===
Trecho_Isolado         | Nova_Rota_Ativa                                                              | Comprimento_Total_m | Tempo_Decisão_ms
-----------------------+------------------------------------------------------------------------------+---------------------+----------------